<a href="https://colab.research.google.com/github/lazarosgogos/ML-exercises/blob/main/ML_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# == FINAL PORJECT - POVERTY PREDICTION CHALLENGE ==

## Load libraries

In [1]:
import pandas as pd
import numpy as np

In [2]:
parent_path = './drive/MyDrive/MSC-ML-exercises/final-project/'
train_df = pd.read_csv(parent_path + 'train_hh_features.csv')
test_df = pd.read_csv(parent_path + 'test_hh_features.csv')
feature_value_desc_df = pd.read_csv(parent_path + 'feature_value_descriptions.csv')
train_df.head()

,hhid,com,weight,strata,utl_exp_ppp17,male,hsize,num_children5,num_children10,num_children18,...,consumed4200,consumed4300,consumed4400,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,survey_id
0,100001,1,75,4,594.80627,Female,1,0,0,0,...,Yes,No,No,No,Yes,Yes,Yes,Yes,No,100000
1,100002,1,150,4,1676.27230,Female,2,0,0,0,...,Yes,No,No,No,No,Yes,Yes,No,No,100000
2,100003,1,375,4,506.93719,Male,5,0,0,2,...,Yes,Yes,No,Yes,Yes,Yes,Yes,No,Yes,100000
3,100004,1,375,4,824.61786,Male,5,0,0,1,...,No,Yes,No,No,No,Yes,Yes,No,No,100000
4,100005,1,525,4,351.47644,Male,7,1,0,0,...,Yes,No,No,Yes,No,Yes,Yes,Yes,No,100000


In [3]:
train_df.describe()

,hhid,com,weight,strata,utl_exp_ppp17,hsize,num_children5,num_children10,num_children18,age,...,share_secondary,sfworkershh,region1,region2,region3,region4,region5,region6,region7,survey_id
count,104234.000000,104234.0,104234.000000,104234.000000,104149.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000,...,104211.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000,104234.000000
mean,222499.501497,1.0,926.357254,4.424813,510.439284,3.611653,0.287872,0.338066,0.571004,52.589414,...,0.228974,0.320479,0.232736,0.157166,0.106395,0.053150,0.235959,0.098653,0.115941,205059.769365
std,83279.120172,0.0,1121.680081,2.429095,437.475542,1.927880,0.561682,0.598281,0.831472,15.732361,...,0.315510,0.400869,0.422577,0.363959,0.308344,0.224333,0.424599,0.298197,0.320156,81587.641825
min,100001.000000,1.0,2.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,15.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100000.000000
25%,126059.250000,1.0,270.000000,2.000000,190.115800,2.000000,0.000000,0.000000,0.000000,41.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100000.000000
50%,219929.500000,1.0,582.000000,5.000000,411.917570,3.000000,0.000000,0.000000,0.000000,52.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,200000.000000
75%,311403.750000,1.0,1128.000000,7.000000,722.440060,5.000000,0.000000,1.000000,1.000000,64.000000,...,0.500000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,300000.000000
max,337462.000000,1.0,23832.000000,8.000000,5880.471200,21.000000,5.000000,6.000000,6.000000,98.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,300000.000000


In [4]:
# train_df.isnull().sum()
train_df['region1']

,region1
0,0
1,0
2,0
3,0
4,0
...,...
104229,0
104230,0
104231,0
104232,0


In [18]:
def to_categorical(df, mapper):
  """ This function assumes the dataframe provided is either the train
  or the test dataframe. It aims to transform the values per column to
  categorical ones, so that the training can take place correctly.
  For example, the `male` column is transformed from Male/Female to 0/1,
  respectively.

  Parameters:
   df: the pandas dataframe

  Returns:
    new_df: a transformed dataframe """

  new_df = df
  for col in df.columns:
    # if df[col].dtype == 'object':
      print(df[col].dtype)
  return new_df
new_df = train_df.copy()
mapper = feature_value_desc_df.copy()

col_maps = {
    col: grp.set_index("Value label")['Value'].to_dict()
    for col, grp in mapper.groupby("Variable name")
}
print(f'length of df {len(new_df)}')
print(f'total nan values: {new_df.isnull().sum().sum()}')
new_df = new_df.dropna()
for col, mapping in col_maps.items():
  if col in new_df.columns:
    new_df[col] = new_df[col].map(mapping).astype('int64')
for col in new_df.columns:
  print(new_df[col].dtype)

length of df 104234


### Filling missing values
Treat each column type differently:
* Binary columns are set to 0, if the value is missing
* Nominal categorical codes are unordered, use the mode (whatever appears most often)
* For counts/discrete quantities, use the median
* For continuous values


In [37]:
new_df = train_df.copy()
mapper = feature_value_desc_df.copy()
col_maps = {
    col: grp.set_index("Value label")['Value'].to_dict()
    for col, grp in mapper.groupby("Variable name")
}

for col, mapping in col_maps.items():
  if col in new_df.columns:
    new_df[col] = new_df[col].map(mapping)

# fill binary values
binary_cols = [
    c for c in new_df.columns
    if c.startswith('consumed') or c in {
        'male', 'owner', "urban","elect","water","toilet",
        "sewer","employed","any_nonagric"
    }
]

new_df[binary_cols] = new_df[binary_cols].fillna(0)

# fill nominal columns
nominal_cols = [
    "dweltyp","educ_max","sector1d",
    "water_source","sanitation_source"
]

for col in nominal_cols:
  if 99 in new_df[col].dropna().unique():
    new_df[col] = new_df[col].fillna(99)
  else:
    new_df[col] = new_df[col].fillna(new_df[col].mode().iloc[0])

# fill counts/discrete quantities
count_cols = [
    "hsize","num_children5","num_children10","num_children18",
    "num_adult_female","num_adult_male","num_elderly","age"
]

for col in count_cols:
  new_df[col] = new_df[col].fillna(new_df[col].median())

# fill continuous values
share_cols = ["sworkershh","sfworkershh","share_secondary"]
new_df[col] = new_df[col].fillna(0)

exp_cols = ["utl_exp_ppp17"] #["cons_ppp17","utl_exp_ppp17"]
for col in exp_cols:
  new_df[col] = new_df[col].fillna(new_df[col].median())

# change from different region columns to single region column
region_cols = [f"region{i}" for i in range(1,8)]

new_df['region'] = (
    new_df[region_cols]
    .idxmax(axis=1)
    .str.replace("region", "", regex=False)
    .astype("Int64")
)
assert (new_df[region_cols].sum(axis=1) == 1).all() # :)
new_df.drop(columns=region_cols, inplace=True)

# for col in new_df.columns:
#   print(col)
# print(new_df['educ_max'])

In [38]:
new_df.corr()

,hhid,com,weight,strata,utl_exp_ppp17,male,hsize,num_children5,num_children10,num_children18,...,consumed4300,consumed4400,consumed4500,consumed4600,consumed4700,consumed4800,consumed4900,consumed5000,survey_id,region
hhid,1.000000,NaN,-0.049521,0.005422,0.008868,-0.021873,-0.034185,-0.029740,-0.020030,-0.019404,...,-0.021028,-0.003325,-0.047968,-0.000118,0.018347,0.002077,0.036179,-0.015782,0.992636,-0.024037
com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,-0.049521,NaN,1.000000,-0.358410,0.298201,0.049485,0.451432,0.197697,0.181805,0.211699,...,0.082791,0.030261,0.093968,0.040934,0.088810,0.109059,0.120600,0.142747,-0.043471,0.200850
strata,0.005422,NaN,-0.358410,1.000000,-0.592770,0.117541,-0.029999,0.036509,0.070243,0.097021,...,-0.122873,-0.013948,-0.118860,-0.224230,-0.289530,-0.195410,-0.262604,-0.025441,0.009229,0.281855
utl_exp_ppp17,0.008868,NaN,0.298201,-0.592770,1.000000,-0.009044,0.214497,0.002027,-0.013155,-0.004083,...,0.150445,0.052676,0.149294,0.218824,0.254620,0.242519,0.246579,0.039096,0.003441,-0.142676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
consumed4800,0.002077,NaN,0.109059,-0.195410,0.242519,-0.008024,0.135280,0.047539,0.045489,0.036083,...,0.216824,0.027423,0.159885,0.193248,0.188666,1.000000,0.144838,0.112196,0.001862,-0.165800
consumed4900,0.036179,NaN,0.120600,-0.262604,0.246579,-0.034813,0.070296,-0.045443,-0.024175,-0.000718,...,0.101313,0.032545,0.093245,0.132690,0.146129,0.144838,1.000000,0.042824,0.029674,-0.123531
consumed5000,-0.015782,NaN,0.142747,-0.025441,0.039096,0.052557,0.302777,0.248762,0.317738,0.229545,...,0.145273,0.029344,0.125034,0.078496,0.108918,0.112196,0.042824,1.000000,-0.016752,0.012280
survey_id,0.992636,NaN,-0.043471,0.009229,0.003441,-0.022856,-0.037767,-0.034988,-0.024371,-0.019325,...,-0.021077,-0.010321,-0.051274,-0.011360,0.013445,0.001862,0.029674,-0.016752,1.000000,-0.012854
